# 财务舞弊识别模型（论文方法重写版）

> 这个 Notebook 是**新文件**，不会改动你原来的 `模型.ipynb`。

## 本版严格遵循的思路
1. 先按年份切分训练集/测试集（防止未来信息泄漏）
2. 只对训练集做欠采样（`RandomUnderSampler`）
3. 以 `Recall`（召回率）最大化为调参目标（`GridSearchCV`）
4. 用最优模型在**原始测试集**上预测与评估

## 你可以把它当成一条主线
- 数据准备 -> 训练集平衡 -> 网格调参 -> 测试评估 -> 保存结果

如果你是第一次看机器学习代码，建议按单元格顺序运行，每个单元格我都加了中文注释。

In [1]:
# ==============================
# 第 1 步：导入依赖库
# ==============================
# 说明：
# - pandas/numpy：数据处理
# - lightgbm：分类模型
# - sklearn：切分、调参、评估
# - imblearn：欠采样（处理类别不平衡）
# - joblib：保存模型

import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, ParameterSampler, StratifiedKFold

warnings.filterwarnings("ignore")

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [2]:
# ==============================
# 第 2 步：读取数据 + 基础检查
# ==============================
# 你只需要保证下面这个文件存在：FS_Preprocessed.xlsx

DATA_PATH = Path("FS_Preprocessed.xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"未找到数据文件: {DATA_PATH.resolve()}")

# 读取 Excel
# 如果你的文件很大，第一次读会稍慢，属于正常现象
df = pd.read_excel(DATA_PATH)

print("数据行数:", len(df))
print("数据列数:", len(df.columns))
print("年份范围:", df["年份"].min(), "-", df["年份"].max())
print("舞弊样本数:", int(df["是否舞弊"].sum()))
print("舞弊比例: {:.2%}".format(df["是否舞弊"].mean()))

# 看前几行，确认字段无误
df.head()

数据行数: 267302
数据列数: 109
年份范围: 2015 - 2025
舞弊样本数: 6842
舞弊比例: 2.56%


,证券代码,证券简称,统计截止日期,年份,是否舞弊,流动资产比率,现金资产比率,应收类资产比率,营运资金对流动资产比率,营运资金比率,...,股权现金流2,股权自由现金流（原有）,全部现金回收率,营运指数,现金适合比率,现金再投资比率,股权自由现金流,企业自由现金流,基本每股收益（披露）,稀释每股收益（披露）
0,1,平安银行,2015-01-01,2015,0,-3.092853,-0.537674,-0.252464,0.105241,-0.417772,...,0.273126,1.214021,-0.113489,0.003466,0.003248,0.000287,2.886134,1.296638,1.280990,1.330761
1,1,平安银行,2015-03-31,2015,0,-3.092853,-0.484837,-0.302568,0.105241,-0.417772,...,1.637346,0.435206,-0.220737,0.005099,0.003411,-0.002652,2.359684,0.423999,0.124043,0.137314
2,1,平安银行,2015-06-30,2015,0,-3.092853,0.061450,-0.332455,0.105241,-0.417772,...,13.566882,0.372859,0.851743,0.046474,0.006065,0.025742,4.375447,0.839272,0.607040,0.635549
3,1,平安银行,2015-09-30,2015,0,-3.092853,0.088316,-0.418599,0.105241,-0.417772,...,14.411648,0.680595,0.273254,0.014986,0.004269,0.009798,9.691865,1.246112,1.090037,1.133784
4,1,平安银行,2015-12-31,2015,0,-3.092853,-0.355877,-0.181263,0.105241,-0.417772,...,4.878768,0.886836,-0.313361,-0.000434,0.002843,-0.004973,10.773664,1.474644,1.415780,1.469803


In [3]:
# ==============================
# 第 3 步：按年份切分训练集/测试集
# ==============================
# 论文口径：
# - 训练集：2015~2022
# - 测试集：2023~2025

train_df = df[df["年份"].between(2015, 2022, inclusive="both")].copy()
test_df = df[df["年份"].between(2023, 2025, inclusive="both")].copy()

# 这些列是“身份信息/标签列”，不参与训练
info_cols = ["证券代码", "证券简称", "统计截止日期", "年份", "是否舞弊"]
feature_cols = [c for c in df.columns if c not in info_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df["是否舞弊"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["是否舞弊"].copy()

print("训练集样本数:", len(X_train), "舞弊比例: {:.2%}".format(y_train.mean()))
print("测试集样本数:", len(X_test), "舞弊比例: {:.2%}".format(y_test.mean()))
print("特征数:", len(feature_cols))

训练集样本数: 190118 舞弊比例: 3.27%
测试集样本数: 77184 舞弊比例: 0.81%
特征数: 104


In [4]:
# ==============================
# 第 4 步：只对训练集做欠采样（核心）
# ==============================
# 重点强调：
# - 欠采样只用于训练集
# - 测试集保持原始分布，才能真实评估模型泛化能力

rus = RandomUnderSampler(random_state=42)
X_train_balanced, y_train_balanced = rus.fit_resample(X_train, y_train)

print("欠采样前训练集样本数:", len(X_train), "舞弊比例: {:.2%}".format(y_train.mean()))
print("欠采样后训练集样本数:", len(X_train_balanced), "舞弊比例: {:.2%}".format(y_train_balanced.mean()))
pd.Series(y_train_balanced).value_counts()

欠采样前训练集样本数: 190118 舞弊比例: 3.27%
欠采样后训练集样本数: 12436 舞弊比例: 50.00%


是否舞弊
0    6218
1    6218
Name: count, dtype: int64

In [5]:
# ==============================
# 第 5 步：极速版分阶段调参（目标 = Recall 最大化）
# ==============================
# 你刚才反馈了“全网格太慢”，这里改成：
# 1) Stage 1 不再重复搜索，直接使用你已跑出的最优结构参数
# 2) Stage 2/3 使用 RandomizedSearchCV（随机搜索）代替全网格
# 3) 交叉验证从 cv=3 调成 cv=2，大幅降低总训练次数
#
# 这样做的核心目标：把整体耗时压到 1~2 小时量级（具体取决于机器性能）。

# 先自动检测 LightGBM 是否能用 GPU：
# - 能用就用 GPU（更快）
# - 不能用就自动回退 CPU（保证代码不报错中断）
def choose_lgb_device(X_sample, y_sample):
    try:
        probe_model = lgb.LGBMClassifier(
            objective="binary",
            boosting_type="gbdt",
            device_type="gpu",
            n_estimators=5,
            max_depth=4,
            num_leaves=15,
            random_state=42,
            verbose=-1,
        )
        probe_model.fit(X_sample, y_sample)
        print("检测结果：GPU 可用，后续调参与训练将使用 GPU。")
        return "gpu"
    except Exception as e:
        print("检测结果：GPU 不可用，自动回退到 CPU。")
        print("原因（截断显示）：", str(e)[:200])
        return "cpu"

# 用一小段数据做探测，开销很小
device_type = choose_lgb_device(
    X_train_balanced.head(2000),
    y_train_balanced.head(2000),
)

base_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "device_type": device_type,
    "random_state": 42,
    "verbose": -1,
}
print("当前 device_type =", device_type)

# ---------- 进度显示配置（你可以按需改） ----------
# SHOW_LIVE_PROGRESS=True：
# - 会打印更细的训练进度（每个候选会有日志）
# - 为了日志顺序清晰，使用单进程 n_jobs=1（速度会慢一点）
#
# SHOW_LIVE_PROGRESS=False：
# - 日志少一些
# - 使用多核并行 n_jobs=-1（更快）
import time
SHOW_LIVE_PROGRESS = False
search_n_jobs = 1 if SHOW_LIVE_PROGRESS else -1
search_verbose = 3 if SHOW_LIVE_PROGRESS else 1

# ---------- 局部搜索策略：围绕论文最优参数 ----------
# 你给出的论文最优参数中心：
# max_depth=10, num_leaves=40, min_child_samples=18, min_child_weight=0.001,
# feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=5,
# reg_alpha=0.02, reg_lambda=2, learning_rate=0.005, n_estimators=500
#
# 为了避免 Stage 1 再跑很久，这里改为“邻域搜索”而不是大范围搜索。

# 为了显示你想要的“4/40”进度，这里不用 sklearn 默认日志，
# 改为手动做参数搜索 + 手动做交叉验证，并在每个 fit 完成后打印进度。
def run_search_with_progress(
    stage_name,
    base_params,
    X,
    y,
    search_space,
    cv=2,
    search_type="grid",
    n_iter=None,
    random_state=42,
):
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, pd.Series):
        y = pd.Series(y)

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    if search_type == "grid":
        candidates = list(ParameterGrid(search_space))
    elif search_type == "random":
        if n_iter is None:
            raise ValueError("random 搜索必须提供 n_iter")
        candidates = list(ParameterSampler(search_space, n_iter=n_iter, random_state=random_state))
    else:
        raise ValueError("search_type 只能是 'grid' 或 'random'")

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    total_fits = len(candidates) * cv
    done_fits = 0

    best_score = -1.0
    best_params = None

    print(f"{stage_name} 开始：候选={len(candidates)}, cv={cv}, 总 fits={total_fits}")
    t_stage = time.time()

    for cand_idx, cand_params in enumerate(candidates, start=1):
        fold_scores = []

        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
            X_tr = X.iloc[tr_idx]
            y_tr = y.iloc[tr_idx]
            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            model = lgb.LGBMClassifier(**base_params, **cand_params)
            model.fit(X_tr, y_tr)

            y_pred = model.predict(X_val)
            fold_recall = recall_score(y_val, y_pred, zero_division=0)
            fold_scores.append(fold_recall)

            done_fits += 1
            print(
                f"{stage_name} 进度: {done_fits}/{total_fits} | "
                f"候选 {cand_idx}/{len(candidates)} | 折 {fold_idx}/{cv} | "
                f"本折Recall={fold_recall:.4f}"
            )

        cand_mean = float(np.mean(fold_scores))
        if cand_mean > best_score:
            best_score = cand_mean
            best_params = cand_params
            print(
                f"{stage_name} 新最优 -> Recall={best_score:.4f}, 参数={best_params}"
            )

    print(f"{stage_name} 完成，用时 {(time.time() - t_stage) / 60:.2f} 分钟")
    print(f"{stage_name} 最优 Recall={best_score:.6f}")
    print(f"{stage_name} 最优参数={best_params}")

    return best_params, best_score


# ---------- Stage 1：只搜结构参数邻域 ----------
stage1_cv = 2
stage1_grid = {
    "max_depth": [8, 9, 10, 11, 12],
    "num_leaves": [31, 40, 50, 63],
}
stage1_best, stage1_best_score = run_search_with_progress(
    stage_name="Stage 1",
    base_params=base_params,
    X=X_train_balanced,
    y=y_train_balanced,
    search_space=stage1_grid,
    cv=stage1_cv,
    search_type="grid",
)

# ---------- Stage 2：只搜正则/采样参数邻域 ----------
stage2_distributions = {
    "min_child_samples": [16, 18, 20],
    "min_child_weight": [0.0008, 0.001, 0.0012],
    "feature_fraction": [0.5, 0.6, 0.7],
    "bagging_fraction": [0.75, 0.8, 0.85],
    "bagging_freq": [4, 5, 6],
    "reg_alpha": [0.01, 0.02, 0.03],
    "reg_lambda": [1.5, 2.0, 3.0],
}
stage2_n_iter = 45
stage2_cv = 2
stage2_best, stage2_best_score = run_search_with_progress(
    stage_name="Stage 2",
    base_params={**base_params, **stage1_best},
    X=X_train_balanced,
    y=y_train_balanced,
    search_space=stage2_distributions,
    cv=stage2_cv,
    search_type="random",
    n_iter=stage2_n_iter,
    random_state=42,
)

# ---------- Stage 3：只搜学习率/迭代轮数邻域 ----------
stage3_distributions = {
    "learning_rate": [0.003, 0.005, 0.008],
    "n_estimators": [400, 500, 700],
}
stage3_n_iter = 9
stage3_cv = 2
stage3_best, stage3_best_score = run_search_with_progress(
    stage_name="Stage 3",
    base_params={**base_params, **stage1_best, **stage2_best},
    X=X_train_balanced,
    y=y_train_balanced,
    search_space=stage3_distributions,
    cv=stage3_cv,
    search_type="random",
    n_iter=stage3_n_iter,
    random_state=42,
)

# ---------- 合并最优参数并重训最终模型 ----------
best_params = {}
best_params.update(stage1_best)
best_params.update(stage2_best)
best_params.update(stage3_best)

print("\n最终合并参数:")
for k, v in best_params.items():
    print(f"{k}: {v}")

# 用合并参数在平衡训练集上训练最终模型
best_model = lgb.LGBMClassifier(**base_params, **best_params)
best_model.fit(X_train_balanced, y_train_balanced)

检测结果：GPU 可用，后续调参与训练将使用 GPU。
当前 device_type = gpu
Stage 1 开始：候选=20, cv=2, 总 fits=40
Stage 1 进度: 1/40 | 候选 1/20 | 折 1/2 | 本折Recall=0.8324
Stage 1 进度: 2/40 | 候选 1/20 | 折 2/2 | 本折Recall=0.8260
Stage 1 新最优 -> Recall=0.8292, 参数={'max_depth': 8, 'num_leaves': 31}
Stage 1 进度: 3/40 | 候选 2/20 | 折 1/2 | 本折Recall=0.8298
Stage 1 进度: 4/40 | 候选 2/20 | 折 2/2 | 本折Recall=0.8241
Stage 1 进度: 5/40 | 候选 3/20 | 折 1/2 | 本折Recall=0.8350
Stage 1 进度: 6/40 | 候选 3/20 | 折 2/2 | 本折Recall=0.8279
Stage 1 新最优 -> Recall=0.8315, 参数={'max_depth': 8, 'num_leaves': 50}
Stage 1 进度: 7/40 | 候选 4/20 | 折 1/2 | 本折Recall=0.8308
Stage 1 进度: 8/40 | 候选 4/20 | 折 2/2 | 本折Recall=0.8331
Stage 1 新最优 -> Recall=0.8319, 参数={'max_depth': 8, 'num_leaves': 63}
Stage 1 进度: 9/40 | 候选 5/20 | 折 1/2 | 本折Recall=0.8231
Stage 1 进度: 10/40 | 候选 5/20 | 折 2/2 | 本折Recall=0.8327
Stage 1 进度: 11/40 | 候选 6/20 | 折 1/2 | 本折Recall=0.8308
Stage 1 进度: 12/40 | 候选 6/20 | 折 2/2 | 本折Recall=0.8273
Stage 1 进度: 13/40 | 候选 7/20 | 折 1/2 | 本折Recall=0.8356
Stage 1 进度: 14/40 | 

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,10
,learning_rate,0.008
,n_estimators,700
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,16


In [6]:
# ==============================
# 第 6 步：在训练集 + 测试集上评估模型
# ==============================
# 说明：
# - 训练集评估：看模型是否学会了规律（是否严重欠拟合）
# - 测试集评估：看模型泛化能力（最重要）
# - 注意：测试集仍然保持原始分布，不做欠采样

def evaluate_binary_classifier(model, X, y, threshold=0.5, set_name="数据集"):
    """给定模型和阈值，输出常用分类指标。"""
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y, y_pred)
    rec = recall_score(y, y_pred)
    pre = precision_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    auc = roc_auc_score(y, y_prob)
    pr_auc = average_precision_score(y, y_prob)

    print(f"\n{'='*20} {set_name} 评估 {'='*20}")
    print(f"阈值: {threshold}")
    print("混淆矩阵:\n", cm)
    print(f"Recall: {rec:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")
    print(f"PR-AUC: {pr_auc:.4f}")

    print("\n分类报告:")
    print(classification_report(y, y_pred, target_names=["非舞弊", "舞弊"], zero_division=0))

    fraud_total = int(np.sum(y == 1))
    fraud_hit = int(cm[1, 1])
    fraud_miss = int(cm[1, 0])
    print("舞弊样本总数:", fraud_total)
    print("成功识别舞弊数:", fraud_hit)
    print("漏判舞弊数:", fraud_miss)

    return y_prob, y_pred, cm, {
        "recall": rec,
        "precision": pre,
        "f1": f1,
        "roc_auc": auc,
        "pr_auc": pr_auc,
        "fraud_total": fraud_total,
        "fraud_hit": fraud_hit,
        "fraud_miss": fraud_miss,
    }

# 默认阈值 0.5：训练集评估（原始训练集分布）
y_train_prob, y_train_pred, cm_train, metrics_train = evaluate_binary_classifier(
    best_model, X_train, y_train, threshold=0.5, set_name="训练集(原始分布)"
)

# 默认阈值 0.5：测试集评估（原始测试集分布）
y_test_prob, y_test_pred, cm_test, metrics_test = evaluate_binary_classifier(
    best_model, X_test, y_test, threshold=0.5, set_name="测试集(原始分布)"
)

# 方便快速对比训练/测试指标
pd.DataFrame([
    {"数据集": "训练集", **metrics_train},
    {"数据集": "测试集", **metrics_test},
])


==================== 训练集(原始分布) 评估 ====================
阈值: 0.5
混淆矩阵:
 [[144485  39415]
 [   373   5845]]
Recall: 0.9400
Precision: 0.1291
F1-score: 0.2271
ROC-AUC: 0.9371
PR-AUC: 0.3715

分类报告:
              precision    recall  f1-score   support

         非舞弊       1.00      0.79      0.88    183900
          舞弊       0.13      0.94      0.23      6218

    accuracy                           0.79    190118
   macro avg       0.56      0.86      0.55    190118
weighted avg       0.97      0.79      0.86    190118

舞弊样本总数: 6218
成功识别舞弊数: 5845
漏判舞弊数: 373

==================== 测试集(原始分布) 评估 ====================
阈值: 0.5
混淆矩阵:
 [[62867 13693]
 [  139   485]]
Recall: 0.7772
Precision: 0.0342
F1-score: 0.0655
ROC-AUC: 0.8833
PR-AUC: 0.0797

分类报告:
              precision    recall  f1-score   support

         非舞弊       1.00      0.82      0.90     76560
          舞弊       0.03      0.78      0.07       624

    accuracy                           0.82     77184
   macro avg       0.52      0.80

,数据集,recall,precision,f1,roc_auc,pr_auc,fraud_total,fraud_hit,fraud_miss
0,训练集,0.940013,0.129143,0.227087,0.937112,0.371460,6218,5845,373
1,测试集,0.777244,0.034208,0.065532,0.883292,0.079698,624,485,139


In [7]:
# ==============================
# 第 7 步：可选 - 阈值扫描
# ==============================
# 说明：
# - 调参目标是 Recall，但实际业务里可以通过阈值再平衡误报/漏报
# - 这里给一个简单扫描，帮助你观察不同阈值下 Recall 的变化

threshold_candidates = [0.30, 0.40, 0.50, 0.60, 0.70]
rows = []

for th in threshold_candidates:
    pred_tmp = (y_test_prob >= th).astype(int)
    rows.append({
        "threshold": th,
        "recall": recall_score(y_test, pred_tmp),
        "precision": precision_score(y_test, pred_tmp, zero_division=0),
        "f1": f1_score(y_test, pred_tmp, zero_division=0),
    })

threshold_df = pd.DataFrame(rows)
threshold_df

,threshold,recall,precision,f1
0,0.3,0.924679,0.021383,0.041799
1,0.4,0.863782,0.027480,0.053266
2,0.5,0.777244,0.034208,0.065532
3,0.6,0.673077,0.042700,0.080306
4,0.7,0.487179,0.056780,0.101706


In [8]:
# ==============================
# 第 8 步：保存模型与结果文件
# ==============================
# 你后续写论文/做复盘时，这几个文件最常用。

# 1) 保存最优模型
model_path = Path("best_lightgbm_model_rewrite.pkl")
joblib.dump(best_model, model_path)
print("模型已保存:", model_path.resolve())

# 2) 导出测试集预测明细（方便人工核查）
result_df = test_df[["证券代码", "证券简称", "年份", "是否舞弊"]].copy()
result_df["预测概率"] = y_test_prob
result_df["预测标签_阈值0.5"] = y_test_pred

result_path = Path("test_predictions_rewrite.xlsx")
result_df.to_excel(result_path, index=False)
print("测试集预测结果已保存:", result_path.resolve())

# 3) 导出特征重要性
fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False)

fi_path = Path("feature_importance_rewrite.csv")
fi_df.to_csv(fi_path, index=False, encoding="utf-8-sig")
print("特征重要性已保存:", fi_path.resolve())

fi_df.head(20)

模型已保存: /home/ubuntu/zxt/MinorPaper/4-模型构建/best_lightgbm_model_rewrite.pkl
测试集预测结果已保存: /home/ubuntu/zxt/MinorPaper/4-模型构建/test_predictions_rewrite.xlsx
特征重要性已保存: /home/ubuntu/zxt/MinorPaper/4-模型构建/feature_importance_rewrite.csv


,feature,importance
9,有形资产比率,1579
8,无形资产比率,1388
11,留存收益资产比,1281
18,母公司所有者权益占比,1213
7,固定资产比率,1193
75,财务费用率,1132
2,应收类资产比率,1109
21,流转税率,1089
91,公司现金流1,1086
1,现金资产比率,1082
